# 2026-07-15 PM—first Stage 1 fit under the v2 DGP

## goal
N_CMA=6 on v2, Stage 2 with PCs, downscale Toronto, Fig 3 with SD > 0.
Reached: Toronto v2 fits, converges, reduces. One reduced curve; five cities open.

## state
- v2 Toronto substrate: n_da 7682, mu [-0.33, 0.655, 0.384], align TRUE, NA 0
- tor_sim: 610,216 deaths, lambda NA/Inf 0, pct zero 98.83
- Stage 1 sliver: variant A wins, qAIC 27,047.0 / B 55,301.9, converged iter 7, coef 25
- red_tor: ref_temp 19.4, theta_star 5, V_star 5x5 pos-def
- fns.R: 338 lines, all 17 pipeline functions, on Drive
- SSOT: six edits pushed — §0 reload order, §0 restore cell, §6.3 prose + fn, §6.3b new, §8.8 stamp
- kernel crashed at Phase 4. Nothing lost.

## what we found
There was never a convergence failure. `sl$cb <- cb` on a data.table dispatches into `set()`, which raises on a 25-column matrix — *Supplied 2868750 items to be assigned to 114750 items of column 'cb'*. `try()` caught it, the wrapper printed "FAILED to converge", the error text went nowhere. gnm was never called. Three sessions of hypotheses ran against a premise no code had tested. The fix is to keep the cross-basis out of the frame and let gnm resolve it from the calling environment: converged at iteration 7, deviance 9,136.5 → 8,042.0, monotone.

## learned
- A function restored from a binary can only be corrected by a `source()` that defines the same name.
- gnm has three outcomes, not two: raises, hits iterMax and returns silently with `$converged` FALSE, or converges.
- The mean is not the median when the tail carries it. Predicted per-DA median 25, came in 7.
- A function's return value is a bank. `run_city` returned substrate + sim + sliver and held two full city sims at once.

## handoff
Phase 4, five cities: MTL, VAN, OTT, CAL, QC. Substrate v2 → sim → sliver → fit → reduce. `run_city` returns `red` and `res` only; `rm` the sim between cities. Toronto regenerates at seed 42 from §5.4 onward.

`mtl_sim_2026-06-24.rds` is additive v1 — re-sim under exp.

OTT/CAL/QC Daymet CSVs: the handoff says verified at 2045/1898/1317 DAs, SSOT §9.8 says drafted. Read the row counts before trusting either.

Phase 5: PCA on CMA-mean vulnerability variables → PC1-3 as meta-regressors → mixmeta. At K=6 `df.residual` should clear the K=3 value of −5, and Q and I² become computable.

Phase 6: downscale, Fig 3. Truth target under the exp DGP is `exp(0.4·F1 + 0.2·F3)`. Success is recovered SD > 0.

§8.8's panel is un-recomputed and stamped as v1 provenance.

Open and unresolved: lambda max 9,759.7 against a lambda0 max of 0.1788. Finite, every check passes, and a DA-day draws 10,000 deaths from a few hundred people. Not a blocker.

phase 0 restore. simulate counts has exp: True, fit_stage1 exists: FALSE, (not fns.R). study dates has tobe 765 (153 warm days x 5 years)

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))

suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(exactextractr); library(terra)
  library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

source(file.path(DRIVE, "fns.R"))

system(sprintf("cd /content && cp %s/saves_eod_2026-06-25.tar.gz . && tar -xzf saves_eod_2026-06-25.tar.gz", DRIVE))
EOD <- "/content/saves_eod"
mtl <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van <- readRDS(file.path(EOD, "van_substrate.rds"))
cma_age_data_mtlvan <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))

year_start <- 2015; year_end <- 2019; warm_months <- 5:9
study_dates <- seq.Date(as.Date(sprintf("%d-05-01", year_start)),
                        as.Date(sprintf("%d-09-30", year_end)), by = "day")
study_dates <- study_dates[lubridate::month(study_dates) %in% warm_months]

sc_body <- paste(deparse(body(simulate_counts)), collapse = " ")
cat("simulate_counts has exp:", grepl("exp(0.4", sc_body, fixed = TRUE), "\n")
cat("fit_stage1 exists:", exists("fit_stage1"), "\n")
cat("fns sourced:", all(sapply(c("read_daymet","build_crossbasis","make_strata_A","make_strata_B",
                                 "qaic","fit_city_sliver","build_city_sim_substrate_v2",
                                 "simulate_counts"), exists)), "\n")
cat("DGP primitives:", all(sapply(c("base_log_rr","lag_weights","L","da_age","annual_rates","daymet_csv"), exists)), "\n")
cat("L dim:", paste(dim(L), collapse=" x "), " study_dates:", length(study_dates), "\n")
cat("packages live — fread:", exists("fread"), " crossbasis:", exists("crossbasis"), "\n")

simulate_counts has exp: TRUE 
fit_stage1 exists: TRUE 
fns sourced: TRUE 
DGP primitives: TRUE 
L dim: 17 x 3  study_dates: 765 
packages live — fread: TRUE  crossbasis: TRUE 


simulate_counts has exp: TRUE
fit_stage1 exists: TRUE
fns sourced: TRUE
DGP primitives: TRUE
L dim: 17 x 3  study_dates: 765
packages live — fread: TRUE  crossbasis: TRUE

---

fit_stage1 exists: TRUE. everything else clean. before phase 1, we need to confirm rm(fit_stage1), that its gone, then define new one into empty slot.

rm the stale fit_stage 1, verify gone then define fresh from SSOT 6.3. no error thrown; TRUE = real fit.

In [ ]:
if (exists("fit_stage1")) rm(fit_stage1)
cat("fit_stage1 gone:", !exists("fit_stage1"), "\n")

fit_stage1 <- function(cma_age_data, cb_template, cma_label, age_label, verbose = TRUE) {
  if (verbose) cat(sprintf("\n=== Stage 1: %s, age %s ===\n", cma_label, age_label))

  cma_age_data[, strata_A := make_strata_A(DA_id, date)]
  cma_age_data[, strata_B := make_strata_B(DA_id, date)]
  cma_age_data[, dow := factor(wday(date))]
  cma_age_data[, t   := as.integer(date - min(date)) + 1]

  results <- list()
  for (variant in c("A", "B")) {
    strata_col <- if (variant == "A") "strata_A" else "strata_B"
    formula <- if (variant == "A") {
      n_deaths ~ cb + dow + ns(t, df = 20)
    } else {
      n_deaths ~ cb + ns(t, df = 20)
    }
    cma_age_data[, strata_use := get(strata_col)]

    fit <- try(gnm(formula, data = cma_age_data,
                   family = quasipoisson(), eliminate = strata_use),
               silent = TRUE)

    if (inherits(fit, "try-error")) {
      err <- conditionMessage(attr(fit, "condition"))
      if (verbose) cat(sprintf("  Variant %s: gnm RAISED -> %s\n", variant, err))
      results[[variant]] <- list(status = "raised", err = err)
      next
    }
    if (!isTRUE(fit$converged)) {
      if (verbose) cat(sprintf("  Variant %s: gnm RAN, converged FALSE (iterMax)\n", variant))
      results[[variant]] <- list(status = "noconv", fit = fit)
      next
    }

    results[[variant]] <- list(
      status   = "ok",
      fit      = fit,
      qaic     = qaic(fit),
      n_strata = length(unique(cma_age_data[[strata_col]])),
      mean_deaths_per_stratum = sum(cma_age_data$n_deaths) /
                                length(unique(cma_age_data[[strata_col]]))
    )
    if (verbose) cat(sprintf("  Variant %s: converged, qAIC = %.1f, strata = %d\n",
                             variant, results[[variant]]$qaic, results[[variant]]$n_strata))
  }

  ok <- names(results)[sapply(results, function(r) r$status == "ok")]
  if (!length(ok)) {
    if (verbose) cat("  BOTH VARIANTS FAILED — returning diagnostics, no fit\n")
    return(invisible(list(cma = cma_label, age = age_label, status = "failed", diag = results)))
  }

  qa <- sapply(ok, function(v) results[[v]]$qaic)
  winner_var <- ok[which.min(qa)]
  wfit <- results[[winner_var]]$fit
  cb_idx <- grep("^cb", names(coef(wfit)))
  stopifnot(length(cb_idx) == 25)

  list(
    cma             = cma_label,
    age             = age_label,
    status          = "ok",
    winner_variant  = winner_var,
    coef            = coef(wfit)[cb_idx],
    vcov            = vcov(wfit)[cb_idx, cb_idx],
    n_strata_A      = if (!is.null(results$A$n_strata)) results$A$n_strata else NA,
    n_strata_B      = if (!is.null(results$B$n_strata)) results$B$n_strata else NA,
    qaic_A          = if (!is.null(results$A$qaic)) results$A$qaic else Inf,
    qaic_B          = if (!is.null(results$B$qaic)) results$B$qaic else Inf,
    mean_dps_winner = results[[winner_var]]$mean_deaths_per_stratum,
    cb_template     = cb_template
  )
}

f1_body <- paste(deparse(body(fit_stage1)), collapse = " ")
cat("fit_stage1 defined:", exists("fit_stage1"), "\n")
cat("has converged test:", grepl("$converged", f1_body, fixed = TRUE), "\n")
cat("has raised branch:", grepl("gnm RAISED", f1_body, fixed = TRUE), "\n")
cat("no canned-string test:", !grepl("FAILED to converge", f1_body, fixed = TRUE), "\n")

fns_path <- file.path(DRIVE, "fns.R")
n_before <- length(readLines(fns_path))
cat("\nfit_stage1 <- ", file = fns_path, append = TRUE)
cat(paste(deparse(fit_stage1), collapse = "\n"), "\n", file = fns_path, append = TRUE)
n_after <- length(readLines(fns_path))
cat("fns.R lines:", n_before, "->", n_after, "\n")
cat("fns.R now carries fit_stage1:", any(grepl("^fit_stage1 <- function", readLines(fns_path))), "\n")

fit_stage1 gone: TRUE 
fit_stage1 defined: TRUE 
has converged test: TRUE 
has raised branch: TRUE 
no canned-string test: TRUE 
fns.R lines: 338 -> 406 
fns.R now carries fit_stage1: TRUE 


fit_stage1 gone: TRUE
fit_stage1 defined: TRUE
has converged test: TRUE
has raised branch: TRUE
no canned-string test: TRUE
fns.R lines: 143 -> 211
fns.R now carries fit_stage1: TRUE

--

stale fit_stage1 removed, fresh one defined into the empty slot. three-way
split confirmed in the body. fns.R 143 -> 211.

read_daymet on toronto csv, build_city_sim_substrate_v2 at
seed 42+35. per-CMA offset mu ~ N(0,0.6) drawn once, then F ~ N(mu,1) per DA.

predict: mu ~ [-0.33, 0.655, 0.384], n_da 7682, align TRUE, NA 0.

In [ ]:
tor_daymet <- read_daymet(daymet_csv)
cat("daymet rows:", nrow(tor_daymet), " DAs:", uniqueN(tor_daymet$DAUID),
    " dates:", uniqueN(tor_daymet$date), "\n")
cat("tmean range:", paste(round(range(tor_daymet$tmean_C, na.rm=TRUE),1), collapse=" to "), "\n")

tor <- build_city_sim_substrate_v2(tor_daymet, da_age, L, seed = 42 + 35) # _v2 not v1; seed = 42 + province prefix, both silent if wrong

cat("\nmu:", paste(round(tor$mu, 3), collapse=" "), "\n") # v1 has no $mu -> errors; reverted patch -> mu≈0 -> false
cat("mu nonzero:", all(abs(tor$mu) > 0.05), "\n")
cat("F means:", paste(round(c(mean(tor$truth_factors$F1),
                             mean(tor$truth_factors$F2),
                             mean(tor$truth_factors$F3)), 3), collapse=" "), "\n")
cat("n_da:", tor$n_da, "\n")
cat("temp_mat:", paste(dim(tor$temp_mat), collapse=" x "), "\n")
cat("align:", all(rownames(tor$temp_mat) == tor$truth_factors$DAUID), "\n")
cat("temp NA:", sum(is.na(tor$temp_mat)),
    " range:", paste(round(range(tor$temp_mat, na.rm=TRUE),1), collapse=" to "), "\n")
cat("Z:", paste(dim(tor$Z), collapse=" x "), " da_age rows:", nrow(tor$da_age), "\n")

daymet rows: 5902740  DAs: 7716  dates: 765 
tmean range: 2.2 to 30.1 

mu: -0.33 0.655 0.384 
mu nonzero: TRUE 
F means: -0.329 0.657 0.388 
n_da: 7682 
temp_mat: 7682 x 765 
align: TRUE 
temp NA: 0  range: 2.2 to 30.1 
Z: 7682 x 17  da_age rows: 7682 


daymet rows: 5902740  DAs: 7716  dates: 765
tmean range: 2.2 to 30.1

mu: -0.33 0.655 0.384
mu nonzero: TRUE
F means: -0.329 0.657 0.388
n_da: 7682
temp_mat: 7682 x 765
align: TRUE
temp NA: 0  range: 2.2 to 30.1
Z: 7682 x 17  da_age rows: 7682

---

mu [-0.33, 0.655, 0.384], F means land on mu not zero = v2 patch is live and between-CMA
variance exists for the first time. 7716 -> 7682 (22 zero-pop + 12 water), and Z/da_age/
truth_factors all carry 7682, so the drop fired in lockstep. align TRUE, NA 0, temp range
unchanged through.

per-DA 80th-pct MMT, rebuild da_long on 7682, fire simulate_counts in exp form.
predict: total deaths ~610,216 read first (v1 was 189,137 — exp(0.3·F2) with mu2=+0.655
inflates 3.2x), rows 23,506,920 (7682 x 765 x 4), NA pop 0, lambda NA/Inf 0, age gradient climbing.

In [ ]:
mmt_tor <- apply(tor$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))
cat("mmt_tor range:", paste(round(range(mmt_tor),1), collapse=" to "),
    " median:", round(median(mmt_tor),1), " NA:", sum(is.na(mmt_tor)), "\n")
cat("mmt aligned to temp_mat:", length(mmt_tor) == nrow(tor$temp_mat), "\n")

tor_da_long <- CJ(da_idx = 1:tor$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(tor$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
tor_da_long <- pop_long[tor_da_long, on = c("da_idx", "age_band")]
tor_da_long[, annual_rate := annual_rates[age_band]]
tor_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

cat("da_long rows:", nrow(tor_da_long), " expect:", tor$n_da * 765 * 4, "\n")
cat("NA pop:", sum(is.na(tor_da_long$pop)),
    " lambda0 range:", paste(round(range(tor_da_long$lambda0),4), collapse=" to "), "\n")

tor_sim <- simulate_counts(tor$temp_mat, tor_da_long, tor$truth_factors, mmt_tor, seed = 42)
# tor$ throughout—mixing tor$temp_mat with v1 truth_factors misaligns every DA silently

cat("\nsim rows:", nrow(tor_sim), "\n")
cat("total deaths:", sum(tor_sim$n_deaths), " (v1 was 189,137)\n")
cat("lambda NA:", sum(is.na(tor_sim$lambda)), " Inf:", sum(is.infinite(tor_sim$lambda)), # exp() overflow check, if this fires it's the DGP not the fitter
    " max:", round(max(tor_sim$lambda, na.rm=TRUE), 4), "\n")
cat("n_deaths NA:", sum(is.na(tor_sim$n_deaths)),
    " pct zero cells:", round(100*mean(tor_sim$n_deaths == 0), 2), "\n")
print(tor_sim[, .(deaths = sum(n_deaths)), by = age_band])

mmt_tor range: 20.6 to 23.6  median: 22.8  NA: 0 
mmt aligned to temp_mat: TRUE 
da_long rows: 23506920  expect: 23506920 
NA pop: 0  lambda0 range: 0 to 0.1788 

sim rows: 23506920 
total deaths: 610216  (v1 was 189,137)
lambda NA: 0  Inf: 0  max: 9759.741 
n_deaths NA: 0  pct zero cells: 98.83 
    age_band deaths
      <char>  <int>
1:  age_0_64  38614
2: age_65_74 161341
3: age_75_84 189439
4:   age_85p 220822


mmt_tor range: 20.6 to 23.6  median: 22.8  NA: 0
mmt aligned to temp_mat: TRUE
da_long rows: 23506920  expect: 23506920
NA pop: 0  lambda0 range: 0 to 0.1788

sim rows: 23506920
total deaths: 610216  (v1 was 189,137)
lambda NA: 0  Inf: 0  max: 9759.741
n_deaths NA: 0  pct zero cells: 98.83
    age_band deaths
      <char>  <int>
1:  age_0_64  38614
2: age_65_74 161341
3: age_75_84 189439
4:   age_85p 220822

--

610,216 deaths, exact to the digit. seed 42 +
v2 substrate is fully deterministic, so 3.2x inflation is DGP's property.
lambda NA/Inf both 0, so the overflow hypothesis is dead. but lambda max 9759.7 against a
lambda0 max of 0.1788—55,000x. base_log_rr 0.03·(T-MMT)^2 hits ~2.7 at 30°C over a 20.6°C
MMT, exp-modulated to log_rr ~8, exp(8) ~ 3000. finite, no NA, every check passes — and a
DA-day drawing 10,000 deaths from a few hundred people. pct zero 98.83 vs v1's 99.2:
deaths concentrated not spread.

150-DA sliver at seed 42, temp attached in row order, cross-basis built, strata stamped.
reading: pct-empty strata, lambda NA/Inf, per-DA death quantiles.

predict: pct empty BELOW 76% (deaths up 3.2x, zero cells 98.83 vs v1 99.2: strata fuller
not emptier), lambda NA/Inf 0, per-DA
deaths wildly skewed (median ~25, max in the thousands), rows 114,750 (150 DAs x 765 days)

In [ ]:
set.seed(42)
idx <- sample(unique(tor_sim$da_idx), 150)
sl  <- tor_sim[da_idx %in% idx & age_band == "age_75_84"]
sl[, DA_id := da_idx]
setorder(sl, da_idx, date)

tl <- data.table(da_idx = rep(idx, each = length(study_dates)),
                 date   = rep(study_dates, times = 150),
                 temp_C = as.vector(t(tor$temp_mat[idx, ]))) # t() then vectorize. column-major without it interleaves DAs and every death gets the wrong day's weather
sl <- tl[sl, on = c("da_idx","date")]

cat("sliver rows:", nrow(sl), " expect:", 150*765, " DAs:", uniqueN(sl$da_idx), "\n")
cat("temp NA:", sum(is.na(sl$temp_C)),
    " range:", paste(round(range(sl$temp_C),1), collapse=" to "), "\n")

sl[, strata_use := make_strata_A(DA_id, date)]

sl_str <- sl[, .(deaths = sum(n_deaths)), by = strata_use]
cat("\nstrata:", nrow(sl_str), " expect:", 150*5*5, "\n")
cat("pct empty:", round(100*mean(sl_str$deaths == 0), 1), "  (v1 baseline 76)\n") # >90 = DGP, drop mu_sd to 0.3. ~76 = fitter, go read gnm raw
cat("mean deaths/stratum:", round(mean(sl_str$deaths), 2), "\n")

cat("\nlambda NA:", sum(is.na(sl$lambda)), " Inf:", sum(is.infinite(sl$lambda)),
    " max:", round(max(sl$lambda, na.rm=TRUE), 2), "\n")

cat("\nper-DA deaths quantiles:\n")
print(quantile(sl[, sum(n_deaths), by = da_idx]$V1, c(0,.25,.5,.75,1)))
cat("sliver total deaths:", sum(sl$n_deaths), "\n")

sliver rows: 114750  expect: 114750  DAs: 150 
temp NA: 0  range: 3.7 to 30.1 

strata: 3750  expect: 3750 
pct empty: 75.4   (v1 baseline 76)
mean deaths/stratum: 0.65 

lambda NA: 0  Inf: 0  max: 71.6 

per-DA deaths quantiles:
  0%  25%  50%  75% 100% 
   0    4    7   13  469 
sliver total deaths: 2419 


sliver rows: 114750  expect: 114750  DAs: 150
temp NA: 0  range: 3.7 to 30.1

strata: 3750  expect: 3750
pct empty: 75.4   (v1 baseline 76)
mean deaths/stratum: 0.65

lambda NA: 0  Inf: 0  max: 71.6

per-DA deaths quantiles:
  0%  25%  50%  75% 100%
   0    4    7   13  469
sliver total deaths: 2419

---

pct empty 75.4 vs v1's 76; branch is fitter not DGP. mu_sd stays 0.6: exp DGP tripled deaths city-wide and moved the empty fraction by 0.6pp,
because the extra deaths landed in strata that already had deaths, not in empty ones.
per-DA quantiles 0/4/7/13/469 — max is 36x the 75th, one DA carries 19% of the sliver's
2,419 deaths. that's the lambda 71.6 DA; same mechanism gave 9,760 city-wide.

predicted
median 25 off the city mean, came in 7—mean isn't median when the tail carries it.

crossbasis on the sliver temp, attach with $<- (:= flattens the 25-col matrix to one vector),
call gnm raw. trace=TRUE prints deviance per iteration.

NaN at iter 1 = response problem,  oscillation = design problem.

predict: converges, lambda finite, strata 75.4% empty same as v1 which fit at qAIC 28,979,
and neither the mu offset nor the exp modulation touches the design matrix.

In [ ]:
cb <- build_crossbasis(sl$temp_C, lag_max = 21)
cat("cb dim:", paste(dim(cb), collapse=" x "), " expect:", nrow(sl), "x 25\n")
cat("cb NA rows:", sum(!complete.cases(cb)), " (expect 21*150 = 3150 lag burn-in)\n")
cat("argvar fun:", attr(cb, "argvar")$fun, " degree:", attr(cb, "argvar")$degree,
    " knots:", paste(round(attr(cb, "argvar")$knots,1), collapse=" "), "\n")

sl$cb  <- cb # base $<- not := — := flattens the 25-col matrix to one vector and formula sees 1 term instead of 25
sl$dow <- factor(wday(sl$date))
sl$t   <- as.integer(sl$date - min(sl$date)) + 1

cat("\ncb survived attach as matrix:", is.matrix(sl$cb),
    " ncol:", ncol(sl$cb), "\n")
cat("t range:", paste(range(sl$t), collapse=" to "), " dow levels:", nlevels(sl$dow), "\n")

cat("\n--- gnm raw, no try() ---\n")
fit_raw <- gnm(n_deaths ~ cb + dow + ns(t, df = 20), # no try(). the error text is the answer here, try() is what hid it for three sessions
               data = sl, family = quasipoisson(),
               eliminate = strata_use,
               verbose = TRUE, trace = TRUE)

cat("\nconverged:", fit_raw$converged, " iter:", fit_raw$iter, "\n")
cat("coef length:", length(coef(fit_raw)), " cb coefs:", length(grep("^cb", names(coef(fit_raw)))), "\n")
cat("coef NA:", sum(is.na(coef(fit_raw))), "\n")
cat("deviance:", deviance(fit_raw), " dispersion:", summary(fit_raw)$dispersion, "\n")

cb dim: 114750 x 25  expect: 114750 x 25
cb NA rows: 21  (expect 21*150 = 3150 lag burn-in)
argvar fun: bs  degree: 2  knots: 12.5 22.2 24.3 


Warning message in set(x, j = name, value = value):
“25 column matrix RHS of := will be treated as one vector”


 {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "Supplied 2868750 items to be assigned to 114750 items of column 'cb'. If you wish to 'recycle' the RHS please use rep() to make this intent clear to readers of your code.",
 .     base::quote(set(x, j = name, value = value)))

---
Error. 2868750 = 114750 x 25. sl is a data.table, so $<- dispatches data.table's method into set(),
which won't take a 25-col matrix in one column slot; raises not flattens.
I thought "$<- never :=" was written for a data.frame.
try() caught this on july 8 and printed "FAILED to converge"; gnm never ran and
there was no convergence problem.

cb stays out of the frame, gnm resolves it from the calling env. model data is a plain
data.frame with vector columns only, cb free-standing 114750x25 alongside. formula binds by
name, rows pair by position off the same sl.

predict: converges (nothing was ever wrong with the fit — v1 fit this exact design at
qAIC 28,979), deviance monotone under 20 iter, cb coefs 25, qAIC 20-30k.1

In [ ]:
mf <- data.frame(
  n_deaths   = sl$n_deaths,
  dow        = factor(wday(sl$date)),
  t          = as.integer(sl$date - min(sl$date)) + 1,
  strata_use = sl$strata_use
  # data.frame not data.table, and cb deliberately absent; gnm resolves cb from the env, rows pair by position off the same sl
)

cat("mf rows:", nrow(mf), " cb rows:", nrow(cb), " match:", nrow(mf) == nrow(cb), "\n") # the only thing pairing cb to mf is row position
cat("cb is matrix:", is.matrix(cb), " ncol:", ncol(cb), "\n")
cat("mf classes:", paste(sapply(mf, function(x) class(x)[1]), collapse=" "), "\n")
cat("n_deaths total:", sum(mf$n_deaths), " (sliver was 2419)\n")
cat("strata levels:", nlevels(droplevels(mf$strata_use)), "\n")

cat("\n--- gnm, cb from calling env ---\n")
fit_raw <- gnm(n_deaths ~ cb + dow + ns(t, df = 20),
               data = mf, family = quasipoisson(),
               eliminate = strata_use, trace = TRUE)

cat("\nconverged:", fit_raw$converged, " iter:", fit_raw$iter, "\n")
cat("coef length:", length(coef(fit_raw)),
    " cb coefs:", length(grep("^cb", names(coef(fit_raw)))), "\n")
cat("coef NA:", sum(is.na(coef(fit_raw))), "\n")
cat("deviance:", round(deviance(fit_raw),1),
    " dispersion:", round(summary(fit_raw)$dispersion,3), "\n")
cat("qAIC:", round(qaic(fit_raw),1), " (v1 sliver A was 28,979)\n")

mf rows: 114750  cb rows: 114750  match: TRUE
cb is matrix: TRUE  ncol: 25
mf classes: integer factor numeric factor
n_deaths total: 2419  (sliver was 2419)
strata levels: 3750

--- gnm, cb from calling env ---
Deviance = 9136.532 Iterations - 1
Deviance = 8196.64 Iterations - 2
Deviance = 8055.676 Iterations - 3
Deviance = 8042.885 Iterations - 4
Deviance = 8042.052 Iterations - 5
Deviance = 8042.012 Iterations - 6
Deviance = 8042.004 Iterations - 7

converged: TRUE  iter: 7
coef length: 51  cb coefs: 25
coef NA: 0
deviance: 8042  dispersion: 0.298
qAIC: 27047  (v1 sliver A was 28,979)

---

converged TRUE, iter 7. deviance 9136.5 -> 8042.0, monotone, flat by iteration 5. cb coefs 25,
coef NA 0, coef length 51 (25 cb + 6 dow + 20 ns(t), no intercept as eliminate replaced it).
qAIC 27,047 sane range, not comparable to v1's 28,979 across different response data.
dispersion 0.298, underdispersed like v1's 0.21, simulated poisson.
the wall was a data.table assignment error, and gnm was never called. three sessions on that.

toronto through fit_stage1 both variants three-outcome status, qAIC pick. data.table with
vector columns only  no cb  column

predict:
  Variant A: converged, qAIC = 27047.0, strata = 3750
  Variant B: converged, qAIC = ~70000, strata = 26250
  winner A, status ok, coef 25, vcov 25x25

In [ ]:
dt <- data.table(
  n_deaths = sl$n_deaths,
  DA_id    = sl$DA_id,
  date     = sl$date
)
cat("dt rows:", nrow(dt), " cb rows:", nrow(cb), " match:", nrow(dt) == nrow(cb), "\n")
cat("dt has cb col:", "cb" %in% names(dt), " (must be false)\n") # cb must not a column as data.table's assignment path raises on a 25-col matrix.
cat("cb in global env:", exists("cb") && is.matrix(cb) && ncol(cb) == 25, "\n")

res_tor <- fit_stage1(dt, cb, "Toronto", "age_75_84") # cb passed as cb_template only not as data

cat("\nstatus:", res_tor$status, " winner:", res_tor$winner_variant, "\n")
cat("qAIC A:", round(res_tor$qaic_A,1), " B:", round(res_tor$qaic_B,1), "\n")
cat("strata A:", res_tor$n_strata_A, " B:", res_tor$n_strata_B, "\n")
cat("coef:", length(res_tor$coef), " vcov:", paste(dim(res_tor$vcov), collapse=" x "),
    " NA:", sum(is.na(res_tor$coef)), "\n")
cat("mean deaths/stratum (winner):", round(res_tor$mean_dps_winner,3), "\n")
cat("cb_template carried:", is.matrix(res_tor$cb_template),
    " argvar:", attr(res_tor$cb_template, "argvar")$fun, "\n")

dt rows: 114750  cb rows: 114750  match: TRUE
dt has cb col: FALSE  (must be false)
cb in global env: TRUE

=== Stage 1: Toronto, age age_75_84 ===
  Variant A: converged, qAIC = 27047.0, strata = 3750
  Variant B: converged, qAIC = 55301.9, strata = 26250

status: ok  winner: A
qAIC A: 27047  B: 55301.9
strata A: 3750  B: 26250
coef: 25  vcov: 25 x 25  NA: 0
mean deaths/stratum (winner): 0.645
cb_template carried: TRUE  argvar: bs

---

A converged at qAIC 27047.0, strata 3750. B converged at 55301.9, strata 26250. A reproduces
block 6 to the digit through the wrapper, fitter doesn't perturb the fit.

both variants
converged, so the raise and iterMax branches stayed untouched.
predicted B ~70000 off v1's 72114, got 55302. same verdict, softer margin: B's penalty is
starved strata, and v2's 3.2x deaths fill them. anchored on v1 without adjusting for the
death count i'd already watched triple

winner A, coef 25, vcov 25x25, NA 0, mean deaths/stratum 0.645. cb_template carries argvar bs.

collapse the lag dim, 25 ->, centered at sliver median with knots at percentiles reference at the 50th.
model.link="log" needed: raw coef/vcov, no model object to read the link off.

predict: ref_temp 19.4 (same daymet, same seed-42 sliver as v1 — the DGP changed deaths not
weather; drift means the sample moved). theta_star 5, V_star 5x5, NA FALSE.

In [ ]:
ref_tor <- median(sl$temp_C) # centering must be the same percentile across cities, not the same temperature, what makes the 5-vectors poolable
cat("ref_temp:", round(ref_tor, 1), " (v1 was 19.4)\n")

red_tor <- reduce_fit(res_tor, ref_tor)

cat("theta_star:", length(red_tor$theta_star),
    " V_star:", paste(dim(red_tor$V_star), collapse=" x "),
    " any NA:", any(is.na(red_tor$theta_star)), "\n")
cat("theta_star:", paste(round(red_tor$theta_star, 3), collapse=" "), "\n")
cat("V_star diag:", paste(round(diag(red_tor$V_star), 4), collapse=" "), "\n")
cat("V_star symmetric:", isTRUE(all.equal(red_tor$V_star, t(red_tor$V_star))), "\n")
cat("V_star pos-def:", tryCatch({ chol(red_tor$V_star); TRUE }, error = function(e) FALSE), "\n") # a non-pos-def V_star here poisons mixmeta's S argument downstream
cat("cma:", red_tor$cma, " age:", red_tor$age, " cen:", round(red_tor$cen, 2), "\n")

ref_temp: 19.4  (v1 was 19.4)
Error in if (model.link %in% c("log", "logit")) {: argument is of length zero
Traceback:

1. crossreduce(basis = cb_t, coef = stage1_result$coef, vcov = stage1_result$vcov,
 .     type = "overall", cen = ref_temp)
2. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "argument is of length zero", base::quote(if (model.link %in%
 .     c("log", "logit")) {
 .     list$RRfit <- exp(fit)
 .     list$RRlow <- exp(fit - z * se)
 .     names(list$RRlow) <- names(fit)
 .     list$RRhigh <- exp(fit + z * se)
 .     names(list$RRhigh) <- names(fit)
 . } else {
 .     list$low <- fit - z * se
 .     names(list$low) <- names(fit)
 .     list$high <- fit + z * se
 .     names(list$high) <- names(fit)
 . }))

---

model.link is length zero. R has no reduce_fit to clobber it with, stale one survived
by being absent from the file. second time today.

rm reduce_fit, retype from SSOT §7.1 with model.link="log", that's the delta, rest is verbatim

In [ ]:
if (exists("reduce_fit")) rm(reduce_fit)
cat("reduce_fit gone:", !exists("reduce_fit"), "\n")

reduce_fit <- function(stage1_result, ref_temp, verbose = TRUE) {
  cb_t <- stage1_result$cb_template
  red  <- crossreduce(
    basis = cb_t,
    coef  = stage1_result$coef,
    vcov  = stage1_result$vcov,
    model.link = "log", # raw coef/vcov, no model objec—crossreduce has no link to read and its error points at model
    type  = "overall",
    cen   = ref_temp
  )
  list(
    cma         = stage1_result$cma,
    age         = stage1_result$age,
    theta_star  = coef(red),
    V_star      = vcov(red),
    cen         = ref_temp,
    reduced_obj = red
  )
}

rf_body <- paste(deparse(body(reduce_fit)), collapse = " ")
cat("reduce_fit defined:", exists("reduce_fit"),
    " has model.link:", grepl("model.link", rf_body, fixed = TRUE), "\n")

fns_path <- file.path(DRIVE, "fns.R")
n_before <- length(readLines(fns_path))
cat("\nreduce_fit <- ", file = fns_path, append = TRUE)
cat(paste(deparse(reduce_fit), collapse = "\n"), "\n", file = fns_path, append = TRUE)
n_after <- length(readLines(fns_path))
cat("fns.R lines:", n_before, "->", n_after, "\n")

fns_txt <- readLines(fns_path)
need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_citait_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive")
audit <- data.table(
  fn      = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt)))
)
audit[, landmine := in_env & !in_file]
print(audit)
cat("\nlandmines:", sum(audit$landmine),
    if (sum(audit$landmine)) paste("->", paste(audit[landmine == TRUE]$fn, collapse=", ")) else "", "\n")

reduce_fit gone: TRUE
reduce_fit defined: TRUE  has model.link: TRUE
fns.R lines: 211 -> 221
                     fn in_env in_file landmine
                 <char> <lgcl>  <lgcl>   <lgcl>
 1:         read_daymet   TRUE    TRUE    FALSE
 2:    build_crossbasis   TRUE    TRUE    FALSE
 3:       make_strata_A   TRUE    TRUE    FALSE
 4:       make_strata_B   TRUE    TRUE    FALSE
 5:                qaic   TRUE    TRUE    FALSE
 6:     fit_city_sliver   TRUE    TRUE    FALSE
 7: build_citait_stage2  FALSE   FALSE    FALSE
 8:          fit_da_pca   TRUE   FALSE     TRUE
 9:    predict_da_theta   TRUE   FALSE     TRUE
10:      compute_da_mmt   TRUE   FALSE     TRUE
11:      monte_carlo_ci   TRUE   FALSE     TRUE
12: standardize_da_rate   TRUE   FALSE     TRUE
13:       save_to_drive  FALSE   FALSE    FALSE

landmines: 5 -> fit_da_pca, predict_da_theta, compute_da_mmt, monte_carlo_ci, standardize_da_rate

---

retype all seven from SSOT verbatim, append to fns.R and re-audit.
after this every function the pipeline calls is in the file, so source() always has something
to clobber with and no frozen binary can pre-empt it.

In [ ]:
for (f in c("fit_da_pca","predict_da_theta","compute_da_mmt","monte_carlo_ci","standardize_da_rate")) {
  if (exists(f)) rm(list = f)
}
cat("landmines cleared:", !any(sapply(c("fit_da_pca","predict_da_theta","compute_da_mmt",
                                        "monte_carlo_ci","standardize_da_rate"), exists)), "\n")

fit_stage2 <- function(reduced_list, cma_predictors_df = NULL,
                       use_pcs = NULL, use_age = NULL, verbose = TRUE) {
  n_obs     <- length(reduced_list)
  theta_mat <- t(sapply(reduced_list, function(x) x$theta_star))
  colnames(theta_mat) <- paste0("theta", seq_len(ncol(theta_mat)))
  V_list    <- lapply(reduced_list, function(x) x$V_star)
  pred_df <- data.table(
    CMA      = sapply(reduced_list, function(x) x$cma),
    age_band = sapply(reduced_list, function(x) x$age),
    obs_idx  = seq_len(n_obs)
  )
  if (!is.null(cma_predictors_df)) pred_df <- merge(pred_df, cma_predictors_df, by = "CMA", all.x = TRUE)
  setorder(pred_df, obs_idx)
  pred_df <- cbind(pred_df, as.data.table(theta_mat))
  n_cma  <- length(unique(pred_df$CMA))
  n_band <- length(unique(pred_df$age_band))
  if (is.null(use_pcs)) use_pcs <- (n_cma > 4 && all(c("PC1","PC2","PC3") %in% names(pred_df)))
  if (is.null(use_age)) use_age <- (n_band > 1)
  rhs <- c(if (use_age) "age_band", if (use_pcs) c("PC1","PC2","PC3"))
  rhs <- if (length(rhs)) paste(rhs, collapse = " + ") else "1"
  lhs <- sprintf("cbind(%s)", paste(colnames(theta_mat), collapse = ", "))
  form <- as.formula(sprintf("%s ~ %s", lhs, rhs))
  use_random <- n_cma >= 3
  fit <- tryCatch(
    mixmeta(form, S = V_list, data = pred_df, method = "reml",
            random = if (use_random) ~ 1 | CMA else NULL),
    error = function(e) { if (verbose) cat("  REML failed:", conditionMessage(e), "-> fixed\n"); NULL }
  )
  method_used <- "reml"
  if (is.null(fit)) {
    fit <- mixmeta(form, S = V_list, data = pred_df, method = "fixed")
    method_used <- "fixed"
  }
  if (verbose) cat(sprintf("  formula: %s | random: %s | method: %s\n",
                           deparse(form), use_random && method_used == "reml", method_used))
  list(fit = fit, pred_df = pred_df, theta_mat = theta_mat, V_list = V_list,
       method = method_used, formula = form)
}

fit_da_pca <- function(Z_matrix, da_ids, verbose = TRUE) {
  pca <- prcomp(Z_matrix, scale. = TRUE)
  var_explained <- summary(pca)$importance["Proportion of Variance", 1:3]
  cum_var       <- summary(pca)$importance["Cumulative Proportion", 3]
  da_scores <- data.table(DAUID = da_ids, PC1 = pca$x[,1], PC2 = pca$x[,2], PC3 = pca$x[,3])
  if (verbose) cat(sprintf("PCA: cum var first 3 = %.1f%%\n", 100*cum_var))
  stopifnot(cum_var > 0.5)
  list(pca = pca, scores = da_scores, var_explained = var_explained)
}

predict_da_theta <- function(stage2_obj, da_ids, band = "age_75_84") {
  cf <- as.numeric(coef(stage2_obj))
  stopifnot(length(cf) == 5)
  data.table(DAUID = da_ids, age_band = band,
             theta1 = cf[1], theta2 = cf[2], theta3 = cf[3], theta4 = cf[4], theta5 = cf[5])
}

compute_da_mmt <- function(da_theta_row, cb_template, temp_range_da, cma_median) {
  av <- attr(cb_template, "argvar")
  red_basis <- onebasis(temp_range_da, fun = av$fun, degree = av$degree, knots = av$knots)
  pred1 <- crosspred(basis = red_basis, coef = da_theta_row, vcov = diag(1e-8, 5),
                     at = temp_range_da, cen = cma_median)
  pred1$predvar[which.min(pred1$allfit)]
}

monte_carlo_ci <- function(stage2_obj, da_scores, n_sim = 1000, verbose = TRUE) {
  beta_hat <- as.vector(coef(stage2_obj$fit))
  V_hat    <- vcov(stage2_obj$fit)
  beta_sims <- MASS::mvrnorm(n_sim, mu = beta_hat, Sigma = V_hat)
  if (verbose) cat(sprintf("MC: %d x %d draws\n", nrow(beta_sims), ncol(beta_sims)))
  beta_sims
}

cdn_2011_std <- c(age_0_64 = 0.8210, age_65_74 = 0.0930, age_75_84 = 0.0580, age_85p = 0.0280)

standardize_da_rate <- function(da_age_rates) {
  rates_wide <- dcast(da_age_rates, DAUID ~ age_band, value.var = "rate")
  std_rate   <- with(rates_wide,
                     cdn_2011_std["age_0_64"]  * age_0_64  +
                     cdn_2011_std["age_65_74"] * age_65_74 +
                     cdn_2011_std["age_75_84"] * age_75_84 +
                     cdn_2011_std["age_85p"]   * age_85p)
  data.table(DAUID = rates_wide$DAUID, std_rate = std_rate)
}

save_to_drive <- function(objs, tag = format(Sys.Date(), "%Y-%m-%d"), drive = DRIVE) {
  dir.create("/content/saves_eod", showWarnings = FALSE)
  for (nm in names(objs)) saveRDS(objs[[nm]], sprintf("/content/saves_eod/%s.rds", nm))
  tarball <- sprintf("saves_eod_%s.tar.gz", tag)
  system(sprintf("cd /content && tar -czf %s saves_eod/ && cp %s '%s/'", tarball, tarball, drive))
  cat("saved", length(objs), "objects ->", file.path(drive, tarball), "\n")
}

new7 <- c("fit_stage2","fit_da_pca","predict_da_theta","compute_da_mmt",
          "monte_carlo_ci","standardize_da_rate","save_to_drive")
n_before <- length(readLines(fns_path))
for (f in new7) {
  cat(sprintf("\n%s <- ", f), file = fns_path, append = TRUE)
  cat(paste(deparse(get(f)), collapse = "\n"), "\n", file = fns_path, append = TRUE)
}
cat('\ncdn_2011_std <- c(age_0_64 = 0.8210, age_65_74 = 0.0930, age_75_84 = 0.0580, age_85p = 0.0280)\n',
    file = fns_path, append = TRUE)
n_after <- length(readLines(fns_path))
cat("fns.R lines:", n_before, "->", n_after, "\n")

fns_txt <- readLines(fns_path)
audit <- data.table(
  fn      = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt)))
)
audit[, landmine := in_env & !in_file]
print(audit)
cat("\nlandmines:", sum(audit$landmine), "\n")
cat("compute_da_mmt has diag(1e-8:",
    grepl("diag(1e-08, 5)", paste(deparse(body(compute_da_mmt)), collapse=" "), fixed = TRUE) ||
    grepl("diag(1e-8, 5)",  paste(deparse(body(compute_da_mmt)), collapse=" "), fixed = TRUE), "\n")

landmines cleared: TRUE
fns.R lines: 221 -> 338
                     fn in_env in_file landmine
                 <char> <lgcl>  <lgcl>   <lgcl>
 1:         read_daymet   TRUE    TRUE    FALSE
 2:    build_crossbasis   TRUE    TRUE    FALSE
 3:       make_strata_A   TRUE    TRUE    FALSE
 4:       make_strata_B   TRUE    TRUE    FALSE
 5:                qaic   TRUE    TRUE    FALSE
 6:     fit_city_sliver   TRUE    TRUE    FALSE
 7: build_citait_stage2  FALSE   FALSE    FALSE
 8:          fit_da_pca   TRUE    TRUE    FALSE
 9:    predict_da_theta   TRUE    TRUE    FALSE
10:      compute_da_mmt   TRUE    TRUE    FALSE
11:      monte_carlo_ci   TRUE    TRUE    FALSE
12: standardize_da_rate   TRUE    TRUE    FALSE
13:       save_to_drive   TRUE    TRUE    FALSE

landmines: 0
compute_da_mmt has diag(1e-8: TRUE

---

incomplete list

rebuild need, print it, re-audit all 17

In [ ]:
need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_city_sim_substrate_v2","simulate_counts",
          "fit_stage1","reduce_fit","fit_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive")
cat("need length:", length(need), "\n")
print(need)

fns_txt <- readLines(fns_path)
audit <- data.table(
  fn      = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt)))
)
audit[, landmine := in_env & !in_file]
print(audit, nrows = 20)
cat("\nrows printed:", nrow(audit), " landmines:", sum(audit$landmine), "\n")
cat("missing from file:", paste(audit[in_file == FALSE]$fn, collapse=", "), "\n")

need length: 17
 [1] "read_daymet"                 "build_crossbasis"           
 [3] "make_strata_A"               "make_strata_B"              
 [5] "qaic"                        "fit_city_sliver"            
 [7] "build_city_sim_substrate_v2" "simulate_counts"            
 [9] "fit_stage1"                  "reduce_fit"                 
[11] "fit_stage2"                  "fit_da_pca"                 
[13] "predict_da_theta"            "compute_da_mmt"             
[15] "monte_carlo_ci"              "standardize_da_rate"        
[17] "save_to_drive"              
                             fn in_env in_file landmine
                         <char> <lgcl>  <lgcl>   <lgcl>
 1:                 read_daymet   TRUE    TRUE    FALSE
 2:            build_crossbasis   TRUE    TRUE    FALSE
 3:               make_strata_A   TRUE    TRUE    FALSE
 4:               make_strata_B   TRUE    TRUE    FALSE
 5:                        qaic   TRUE    TRUE    FALSE
 6:             fit_city_sliver   TRUE    TRUE    FALSE
 7: build_city_sim_substrate_v2   TRUE    TRUE    FALSE
 8:             simulate_counts   TRUE    TRUE    FALSE
 9:                  fit_stage1   TRUE    TRUE    FALSE
10:                  reduce_fit   TRUE    TRUE    FALSE
11:                  fit_stage2   TRUE    TRUE    FALSE
12:                  fit_da_pca   TRUE    TRUE    FALSE
13:            predict_da_theta   TRUE    TRUE    FALSE
14:              compute_da_mmt   TRUE    TRUE    FALSE
15:              monte_carlo_ci   TRUE    TRUE    FALSE
16:         standardize_da_rate   TRUE    TRUE    FALSE
17:               save_to_drive   TRUE    TRUE    FALSE

rows printed: 17  landmines: 0
missing from file:

---

every function the pipeline calls is now in fns.R. three sessions.

re-run the reduce against the fixed reduce_fit. predict ref_temp 19.4, theta_star 5,
V_star 5x5 symmetric pos-def, NA FALSE.

In [ ]:
ref_tor <- median(sl$temp_C) # centering is the same percentile across cities, not the same temperature; what makes the 5-vectors poolable
cat("ref_temp:", round(ref_tor, 1), " (v1 was 19.4)\n")

red_tor <- reduce_fit(res_tor, ref_tor)

cat("theta_star:", length(red_tor$theta_star),
    " V_star:", paste(dim(red_tor$V_star), collapse=" x "), # a non-pos-def V_star poisons mixmeta's S downstream
    " any NA:", any(is.na(red_tor$theta_star)), "\n")
cat("theta_star:", paste(round(red_tor$theta_star, 3), collapse=" "), "\n")
cat("V_star diag:", paste(round(diag(red_tor$V_star), 4), collapse=" "), "\n")
cat("V_star symmetric:", isTRUE(all.equal(red_tor$V_star, t(red_tor$V_star))), "\n")
cat("V_star pos-def:", tryCatch({ chol(red_tor$V_star); TRUE }, error = function(e) FALSE), "\n")
cat("cma:", red_tor$cma, " age:", red_tor$age, " cen:", round(red_tor$cen, 2), "\n")

ref_temp: 19.4  (v1 was 19.4)
theta_star: 5  V_star: 5 x 5  any NA: FALSE
theta_star: -0.843 -6.486 -5.099 -6.053 -3.616
V_star diag: 3.9849 1.8988 2.9041 4.0954 37.4479
V_star symmetric: TRUE
V_star pos-def: TRUE
cma: Toronto  age: age_75_84  cen: 19.38

---

ref_temp 19.4, reproduces v1. theta_star 5, V_star 5x5, symmetric, pos-def, NA FALSE.
theta_star [-0.843, -6.486, -5.099, -6.053, -3.616]: basis coefficients not RRs; they only
mean something through the basis at a temperature, so the signs don't read on their own.
V_star diag [3.98, 1.90, 2.90, 4.10, 37.45], 5th is 10x the rest. it's the top B-spline
term above the 90th-pct knot where there's little data. mixmeta down-weights it by inverse vcov.

first v2 reduced curve. one city down, five to go!